In [ ]:

# ==========================================================
# PROJECT 2 - STARTUP INVESTMENT ANALYTICS
# REAL DATA | GOOGLE COLAB | SINGLE CELL
# ==========================================================

import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "pandas", "numpy", "plotly", "requests"
])

import pandas as pd
import numpy as np
import plotly.express as px
import requests
import re
import zipfile
from pathlib import Path
from io import BytesIO
from plotly.io import to_html
from IPython.display import display

# 1. SETTINGS
PROJECT = Path("/content/Startup_Investment_Analytics")
PROJECT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ZIP = False

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "NishantT2001/Indian_Startup_Funding_Analysis/"
    "main/startup_dataset.csv"
)

# 2. DOWNLOAD REAL DATA
print("Downloading real startup funding dataset...")

response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()

raw = pd.read_csv(
    BytesIO(response.content),
    encoding="utf-8-sig",
    low_memory=False
)

print("Dataset loaded!")
print("Original records:", len(raw))
print("Original columns:", list(raw.columns))

# 3. STANDARDIZE COLUMNS
def normalize(name):
    return re.sub(
        r"[^a-z0-9]+", "_", str(name).lower()
    ).strip("_")

raw.columns = [normalize(c) for c in raw.columns]

aliases = {
    "Date": ["date_dd_mm_yyyy", "date"],
    "Startup": ["startup_name", "startupname"],
    "Industry": [
        "industry_vertical", "industry", "sector"
    ],
    "City": [
        "city_location", "city", "location"
    ],
    "Investors": [
        "investors_name", "investors",
        "investor_name"
    ],
    "Funding_Type": [
        "investmentntype", "investment_type",
        "funding_type"
    ],
    "Amount_USD": [
        "amount_in_usd", "amountinusd",
        "amount_usd"
    ]
}

mapping = {}

for standard, names in aliases.items():
    for name in names:
        if name in raw.columns:
            mapping[name] = standard
            break

df = raw.rename(columns=mapping).copy()

required = ["Startup", "Amount_USD"]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        f"Available: {list(df.columns)}"
    )

# 4. CLEANING
for col in [
    "Startup", "Industry", "City",
    "Investors", "Funding_Type"
]:
    if col in df.columns:
        df[col] = (
            df[col].astype("string")
            .str.strip()
            .replace("", pd.NA)
        )

df["Amount_USD"] = (
    df["Amount_USD"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)

df["Amount_USD"] = pd.to_numeric(
    df["Amount_USD"], errors="coerce"
)

df.loc[
    df["Amount_USD"] < 0, "Amount_USD"
] = np.nan

if "Date" in df.columns:
    date_text = (
        df["Date"].astype("string")
        .str.strip()
        .str.replace(".", "/", regex=False)
        .str.replace("-", "/", regex=False)
    )

    df["Date"] = pd.to_datetime(
        date_text,
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Year_Month"] = df["Date"].dt.strftime("%Y-%m")

df = df.dropna(subset=["Startup"])
df = df.drop_duplicates().reset_index(drop=True)

if df.empty:
    raise ValueError("No usable startup records.")

df.to_csv(
    PROJECT / "cleaned_startup_data.csv",
    index=False
)

print("\nCLEANED DATA")
display(df.head())

# Funding analysis uses disclosed numeric amounts only
funded = df.dropna(subset=["Amount_USD"]).copy()

if funded.empty:
    raise ValueError("No numeric funding amounts available.")

# 5. KPIs
kpis = {
    "Total Funding Records": len(df),
    "Records With Disclosed Amount": len(funded),
    "Unique Startups": df["Startup"].nunique(),
    "Total Disclosed Funding USD":
        funded["Amount_USD"].sum(),
    "Average Disclosed Funding USD":
        funded["Amount_USD"].mean(),
    "Largest Disclosed Round USD":
        funded["Amount_USD"].max()
}

for col in ["Industry", "City", "Investors"]:
    if col in df.columns:
        kpis["Unique " + col] = df[col].nunique()

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

print("\nSTARTUP INVESTMENT KPIs")
display(kpi_df)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

# 6. INTERACTIVE CHARTS
figures = []

def chart(fig, title):
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(l=45, r=40, t=75, b=55)
    )
    figures.append((title, fig))
    fig.show()

def aggregate(group, metric="Amount_USD", top=15):
    result = (
        funded.dropna(subset=[group])
        .groupby(group, as_index=False)[metric]
        .sum()
    )
    return result.nlargest(top, metric)

# CHART 1 - TOP STARTUPS
d = aggregate("Startup").sort_values("Amount_USD")

chart(
    px.bar(
        d, x="Amount_USD", y="Startup",
        orientation="h", color="Amount_USD"
    ),
    "1. Top 15 Startups by Disclosed Funding"
)

# CHART 2 - INDUSTRY FUNDING
if "Industry" in df.columns:
    d = aggregate("Industry").sort_values("Amount_USD")

    chart(
        px.bar(
            d, x="Amount_USD", y="Industry",
            orientation="h", color="Amount_USD"
        ),
        "2. Top Industries by Funding"
    )

# CHART 3 - CITY FUNDING
if "City" in df.columns:
    d = aggregate("City").sort_values("Amount_USD")

    chart(
        px.bar(
            d, x="Amount_USD", y="City",
            orientation="h", color="Amount_USD"
        ),
        "3. Top Cities by Funding"
    )

# CHART 4 - YEARLY FUNDING
if "Year" in df.columns:
    d = (
        funded.dropna(subset=["Year"])
        .groupby("Year", as_index=False)["Amount_USD"]
        .sum().sort_values("Year")
    )

    chart(
        px.line(
            d, x="Year", y="Amount_USD",
            markers=True
        ),
        "4. Yearly Disclosed Funding Trend"
    )

# CHART 5 - FUNDING TYPE
if "Funding_Type" in df.columns:
    d = aggregate("Funding_Type", top=20)

    chart(
        px.pie(
            d, names="Funding_Type",
            values="Amount_USD", hole=0.4
        ),
        "5. Funding by Investment Type"
    )

# CHART 6 - TOP INVESTOR LABELS
if "Investors" in df.columns:
    # Original cells may contain multiple investors.
    # This chart counts exact reported investor labels.
    d = (
        df.dropna(subset=["Investors"])
        .groupby("Investors")
        .size()
        .reset_index(name="Records")
        .nlargest(15, "Records")
        .sort_values("Records")
    )

    chart(
        px.bar(
            d, x="Records", y="Investors",
            orientation="h", color="Records"
        ),
        "6. Most Frequent Reported Investor Labels"
    )

# CHART 7 - FUNDING DISTRIBUTION
chart(
    px.histogram(
        funded,
        x="Amount_USD",
        nbins=50,
        log_y=True
    ),
    "7. Distribution of Disclosed Funding Amounts"
)

# CHART 8 - YEAR VS INDUSTRY
if {"Year", "Industry"}.issubset(df.columns):
    d = funded.dropna(
        subset=["Year", "Industry"]
    ).copy()

    top_industries = (
        d.groupby("Industry")["Amount_USD"]
        .sum().nlargest(10).index
    )

    d = d[d["Industry"].isin(top_industries)]

    if not d.empty:
        matrix = d.pivot_table(
            index="Industry",
            columns="Year",
            values="Amount_USD",
            aggfunc="sum",
            fill_value=0
        )

        chart(
            px.imshow(
                matrix,
                aspect="auto",
                labels={"color": "Funding USD"}
            ),
            "8. Industry vs Year Funding Heatmap"
        )

# 7. CSV REPORTS
for group in [
    "Startup", "Industry", "City",
    "Funding_Type", "Year", "Year_Month"
]:
    if group not in funded.columns:
        continue

    report = (
        funded.groupby(group, dropna=False)["Amount_USD"]
        .agg(
            Total_Funding_USD="sum",
            Average_Funding_USD="mean",
            Disclosed_Records="count"
        )
        .reset_index()
    )

    report.to_csv(
        PROJECT / f"summary_{group.lower()}.csv",
        index=False
    )

if "Investors" in df.columns:
    (
        df.groupby("Investors", dropna=False)
        .size()
        .reset_index(name="Records")
        .to_csv(
            PROJECT / "summary_investors.csv",
            index=False
        )
    )

# 8. COMBINED DASHBOARD
html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Startup Investment Analytics</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p { text-align: center; }
.chart {
    background: white;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
    border-radius: 12px;
}
</style>
</head>
<body>
<h1>STARTUP INVESTMENT ANALYTICS</h1>
<p>Historical Indian Startup Funding Data</p>
"""

for i, (title, fig) in enumerate(figures):
    html += (
        '<div class="chart">'
        + to_html(
            fig,
            full_html=False,
            include_plotlyjs=(i == 0)
        )
        + "</div>"
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Startup_Investment_Dashboard.html"
)

dashboard_path.write_text(
    html, encoding="utf-8"
)

# 9. README
readme = f"""
PROJECT 2 - STARTUP INVESTMENT ANALYTICS

Data source:
{DATA_URL}

Original Kaggle dataset:
https://www.kaggle.com/datasets/sudalairajkumar/indian-startup-funding

Clean records: {len(df)}
Disclosed funding records: {len(funded)}
Charts: {len(figures)}

No synthetic funding records generated.

Limitations:
Undisclosed funding amounts are excluded
from monetary totals and averages.
Investor labels can contain multiple names.
Funding data is historical, not live.
"""

(PROJECT / "README.txt").write_text(
    readme, encoding="utf-8"
)

# 10. ZIP
zip_path = Path(
    "/content/Startup_Investment_Analytics.zip"
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for file in PROJECT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )

print("\n" + "=" * 45)
print("PROJECT 2 COMPLETED")
print("=" * 45)
print("Clean records:", len(df))
print("Charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
